# Day 6 — Exception Hierarchies & Error Design

## Objective
Demonstrate custom exception design (`PipelineError` hierarchy), user-facing error messages (WHAT, WHERE, WHY), top-level error handling, exception chaining (`raise ... from ...`), and `try/except/else/finally` control flow semantics.

## 1. Custom Exception Hierarchy

```text
Exception
└── PipelineError
    ├── ConfigError
    ├── DataValidationError
    └── ProcessingError
```

In [1]:
import sys
from pathlib import Path
repo_root = Path.cwd().resolve()
sys.path.insert(0, str(repo_root / 'src'))

from task_analytics import (
    PipelineError, ConfigError, DataValidationError, ProcessingError, validate_pipeline_config, CSVBatchIterator, NormalizeDataStep
)
print('Custom exception hierarchy components loaded successfully.')

ImportError: cannot import name 'validate_pipeline_config' from 'task_analytics' (C:\Users\bc\task-management-api\src\task_analytics\__init__.py)


## 2. Triggering Specific Error Failure Modes

Catch each error at top level using `except PipelineError:`.

In [2]:
print('--- 1. Triggering ConfigError ---')
try:
    validate_pipeline_config({'data_path': 'data', 'batch_size': -5})
except PipelineError as e:
    print(f'Caught {type(e).__name__}: {e}')

print('\n--- 2. Triggering DataValidationError ---')
try:
    CSVBatchIterator('data/non_existent_file_999.csv')
except PipelineError as e:
    print(f'Caught {type(e).__name__}: {e}')

print('\n--- 3. Triggering ProcessingError with Exception Chaining ---')
step = NormalizeDataStep(numeric_fields=['estimated_hours'])
try:
    step.execute([{'task_id': 'TASK-1', 'estimated_hours': 'INVALID_FLOAT'}])
except PipelineError as e:
    print(f'Caught {type(e).__name__}: {e}')
    if e.__cause__:
        print('  Chained Cause preserved:', type(e.__cause__).__name__, '->', e.__cause__)

--- 1. Triggering ConfigError ---
NameError: name 'validate_pipeline_config' is not defined


## 3. `try / except / else / finally` Control Flow Semantics

In [3]:
print('--- Case A: Success Path ---')
try:
    res = 10 / 2
except ZeroDivisionError:
    print('except executed')
else:
    print(f'else executed (success!): result = {res}')
finally:
    print('finally executed (always)')

print('\n--- Case B: Failure Path ---')
try:
    res = 10 / 0
except ZeroDivisionError as e:
    print('except executed: caught zero division')
else:
    print('else executed')
finally:
    print('finally executed (always)')

--- Case A: Success Path ---
else executed (success!): result = 5.0
finally executed (always)

--- Case B: Failure Path ---
except executed: caught zero division
finally executed (always)


## Conclusion & Key Takeaways

- **Specific Exceptions**: Derive custom exceptions from a single base class `PipelineError` for clean top-level handling.
- **Exception Chaining**: Use `raise ProcessingError(...) from err` to preserve original tracebacks for debugging.